In [1]:
%%writefile check.py
import os
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
import polars as pl
from scipy.special import softmax

SYS_PRMPT = """
You are an experienced, fair, unbiased moderator. 
Classify whether the comment violates the supplied moderation rule.
- True: the comment breaks the specified rule
- False: the comment is considered safe in relation to the specified rule
Respond only using True or False.
""".strip()

USR_PRMPT_TMPLT = """
[RULE]: {}
[True EXAMPLE]: {}
[False EXAMPLE]: {}
[True EXAMPLE 2]: {}
[False EXAMPLE 2]: {}
[TEST CASE COMMENT]: {}
""".strip()

CHOICES = ['True', 'False']

def chat_formatting(df, tokenizer):
  prompts = []
  for user_content in df['user_content']:
    chat = [
      {'role': 'system', 'content': SYS_PRMPT},
      {'role': 'user', 'content': user_content.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
      chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    prompts.append(prompt)
  df = df.with_columns(pl.Series('prompt', prompts))
  return df


if __name__ == '__main__':
  print('importing vllm...')
  import vllm
  from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
  print('loading vllm...')

  llm = vllm.LLM(
    '/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1',
    quantization='awq',
    task='generate',
    tensor_parallel_size=2,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    enable_prefix_caching=True,
    dtype='half',
    enforce_eager=True,
    disable_log_stats=True,
    disable_custom_all_reduce=True,
  )
  print('building prompts...')
  path = '/kaggle/input/jigsaw-agile-community-rules/test.csv' if IS_SUB else '/kaggle/input/jigsaw-agile-community-rules/train.csv'
  df = (
      pl.read_csv(path)
      .with_columns([pl.col(x).str.replace_all(r'\s+', ' ') for x in ["body", "^.*_example_.*$"]])
      .with_columns(pl.format(USR_PRMPT_TMPLT,  "rule", "positive_example_1", "negative_example_1","positive_example_2", "negative_example_2", "body").alias('user_content'))
  )

  tokenizer = llm.get_tokenizer()
  df = chat_formatting(df, tokenizer)
  prompts = df['prompt'].to_list()
  mclp = MultipleChoiceLogitsProcessor(
    tokenizer,
    choices=CHOICES,
  )
  sampling_params_choice = vllm.SamplingParams(seed=1337, skip_special_tokens=True, max_tokens=1, logits_processors=[mclp], logprobs=len(mclp.choices),)
  outputs = llm.generate(prompts, sampling_params_choice, use_tqdm=True)
  logprobs = [
    {lp.decoded_token: lp.logprob for lp in list(lps)}
    for lps in [output.outputs[0].logprobs[0].values() for output in outputs]
  ]
  choices = [max(d, key=d.get) for d in logprobs]
  print('generation end')
  df = df.with_columns(pl.Series('logprobs', logprobs), pl.Series('type', choices))
  print(df.group_by('type').agg(pl.len()).sort(['type']))
  logprobs = df['logprobs'].to_numpy()
  probs = softmax(logprobs, axis=-1)
  sub = df.with_columns(pl.Series('rule_violation', probs[:,0].tolist()))
  sub.select('row_id', 'rule_violation').write_csv('submission.csv')
  if not IS_SUB:
    import pandas as pd
    from sklearn.metrics import roc_auc_score
    
    # Load CSVs
    submission = pd.read_csv("submission.csv")
    gt = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")[['row_id', 'rule_violation']]
    
    # Merge on 'row_id' to align predictions and ground truth
    merged = pd.merge(gt, submission, on="row_id", suffixes=('_gt', '_pred'))
    
    # Ensure proper columns
    y_true = merged['rule_violation_gt']
    y_score = merged['rule_violation_pred']
    
    # Compute AUC
    try:
        auc = roc_auc_score(y_true, y_score)
        print(f"Column-Averaged AUC: {auc:.6f}")
    except ValueError as e:
        print(f"Cannot compute AUC: {e}")

Writing check.py


In [2]:
 %%bash
# # WHEELHOUSE=/kaggle/input/mdc-wheelhouse/wheelhouse
uv pip uninstall --system 'tensorflow'
# uv pip install -U --system --no-index --find-links=$WHEELHOUSE 'polars==1.31.0' 'vllm' 'triton' 'numpy<2' 'logits-processor-zoo'

Using Python 3.11.13 environment at: /usr
Uninstalled 1 package in 1.79s
 - tensorflow==2.18.0


In [3]:
! VLLM_USE_V1=0 TORCH_CUDA_ARCH_LIST=7.5 python check.py

importing vllm...
loading vllm...
INFO 09-12 00:52:58 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 09-12 00:53:11 [config.py:1604] Using max model len 4096
WARNING 09-12 00:53:12 [config.py:1084] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 09-12 00:53:13 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-12 00:53:13 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', speculative_config=None, tokenizer='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUT

In [4]:
# ! VLLM_USE_V1=0 TORCH_CUDA_ARCH_LIST=7.5 python check1.py

In [5]:
! head -n 5 submission.csv

row_id,rule_violation
0,0.9999998509151202
1,0.9999865799848968
2,2.2159489282323004e-8
3,0.9999999645892373
